## Autoregressive Models

#### Predicting S&P500

In [50]:
import numpy as np
import pandas as pd
from datetime import datetime
import statsmodels.api as sm
import yfinance as yf
import plotly.graph_objects as go
from datetime import datetime

##### Simulating AR(1) process

In [46]:
np.random.seed(42)
n = 200
phi = 0.72
c = 0.5 # intercept or constant
errors = np.random.normal(0, 1, n)
y = np.zeros(n)

for t in range(1,n):
    y[t] = c + phi * y[t-1] + errors[t]

fig1 = go.Figure()

fig1.add_trace(go.Scatter(
    x = np.arange(n),
    y = y,
    mode = 'lines+markers',
))

fig1.update_layout(
    title = 'AR examples_ random generated data',
    xaxis_title = 'timestep',
    yaxis_title = 'price',
    template = 'plotly_white',
    width = 960,
    height = 450
)
fig1.show()

##### Fit AR Model

In [26]:
model = sm.tsa.ARIMA(y, order=(1,0,0)) #AR 1
results = model.fit()
print(results.summary())

                               SARIMAX Results                                
Dep. Variable:                      y   No. Observations:                  200
Model:                 ARIMA(1, 0, 0)   Log Likelihood                -269.339
Date:                Sat, 09 Aug 2025   AIC                            544.677
Time:                        18:49:53   BIC                            554.572
Sample:                             0   HQIC                           548.682
                                - 200                                         
Covariance Type:                  opg                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          1.5898      0.195      8.146      0.000       1.207       1.972
ar.L1          0.6637      0.055     12.090      0.000       0.556       0.771
sigma2         0.8629      0.087      9.919      0.0

##### Applying to S&P500 returns

In [43]:
spy_data = yf.download('^GSPC', start='2019-01-01', end='2025-06-30', auto_adjust=True)
spy_data['returns'] = spy_data['Close'].pct_change()
spy_data.dropna(inplace=True)
spy_data.tail(7)

[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume,returns
Ticker,^GSPC,^GSPC,^GSPC,^GSPC,^GSPC,
Date,,,,,,
2025-06-18,5980.870117,6018.250000,5971.890137,5987.930176,5106470000,-0.000309
2025-06-20,5967.839844,6018.200195,5952.560059,5999.669922,7451500000,-0.002179
2025-06-23,6025.169922,6028.770020,5943.229980,5969.669922,5597000000,0.009607
2025-06-24,6092.180176,6101.759766,6059.250000,6061.209961,5443690000,0.011122
2025-06-25,6092.160156,6108.509766,6080.089844,6104.229980,5171110000,-0.000003
2025-06-26,6141.020020,6146.520020,6107.270020,6112.089844,5308140000,0.008020
2025-06-27,6173.069824,6187.680176,6132.350098,6150.700195,7889350000,0.005219


In [59]:
# plot returns
fig2 = go.Figure()

fig2.add_trace(go.Scatter(
    x = spy_data.index,
    y = spy_data['returns'],
    mode = 'lines',
))

fig2.update_layout(
    title = f'SPY Daily returns plot {spy_data.index[0].strftime('%B_%Y')} and {spy_data.index[-1].strftime('%B_%Y')}',
    xaxis = {'title': 'timestep'},
    yaxis = {'title': 'price', 'tickformat': '.2%'},
    template = 'plotly_white',
    width = 960,
    height = 450
)
fig2.show()

###### Fit AR(1) to returns

In [61]:
model_sp = sm.tsa.ARIMA(spy_data['returns'].values, order=(1,0,0))
results_sp = model_sp.fit()

print(results_sp.summary())

                               SARIMAX Results                                
Dep. Variable:                      y   No. Observations:                 1630
Model:                 ARIMA(1, 0, 0)   Log Likelihood                4801.882
Date:                Sat, 09 Aug 2025   AIC                          -9597.765
Time:                        19:09:48   BIC                          -9581.576
Sample:                             0   HQIC                         -9591.759
                               - 1630                                         
Covariance Type:                  opg                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0006      0.000      2.293      0.022     9.2e-05       0.001
ar.L1         -0.1698      0.010    -16.921      0.000      -0.189      -0.150
sigma2         0.0002   2.26e-06     71.505      0.0

##### Forecast next 10 days

In [64]:
forecast_sp = results_sp.predict(n_periods=10)
forecast_sp

array([ 0.00063333,  0.004944  , -0.0050886 , ..., -0.00114737,
        0.00074141, -0.00062079], shape=(1630,))

In [65]:
forecast_sp2 = results_sp.forecast(steps=10)
forecast_sp2

array([-0.00014522,  0.00076551,  0.00061089,  0.00063714,  0.00063268,
        0.00063344,  0.00063331,  0.00063333,  0.00063333,  0.00063333])

In [67]:
print(forecast_sp2.shape)

(10,)
